In [2]:
import json
import pandas as pd
from collections import Counter
import re

# CONFIG

INPUT_FILE = "gov_data_be.json"

# LOAD DATA

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    datasets = json.load(f)

print(f"Datasets loaded: {len(datasets)}")

# COLLECT KEYWORDS

all_keywords = []

for dataset in datasets:

    keywords = dataset.get(
        "keywords",
        []
    )

    for kw in keywords:

        kw = kw.strip()

        if kw:

            all_keywords.append(kw)

print(
    f"Total keyword occurrences: "
    f"{len(all_keywords)}"
)

# FREQUENCY ANALYSIS
keyword_counts = Counter(
    all_keywords
)

print(
    f"Unique keywords: "
    f"{len(keyword_counts)}"
)

# MOST FREQUENT KEYWORDS

print("\nTOP 100 KEYWORDS\n")

for kw, freq in keyword_counts.most_common(100):

    print(
        f"{freq:5d}  {kw}"
    )

# SAVE FULL FREQUENCY TABLE

df_freq = pd.DataFrame(
    keyword_counts.items(),
    columns=[
        "keyword",
        "frequency"
    ]
)

df_freq = df_freq.sort_values(
    by="frequency",
    ascending=False
)

df_freq.to_csv(
    "keyword_frequencies.csv",
    index=False,
    encoding="utf-8"
)

print(
    "\nSaved: keyword_frequencies.csv"
)

# FIND SUSPICIOUS KEYWORDS

def contains_non_ascii(text):

    return any(
        ord(ch) > 127
        for ch in text
    )

def contains_accent(text):

    return bool(
        re.search(
            r"[àáâãäåèéêëìíîïòóôõöùúûüçñ]",
            text.lower()
        )
    )

suspicious = []

for kw, freq in keyword_counts.items():

    reason = []

    if contains_non_ascii(kw):
        reason.append(
            "non_ascii"
        )

    if contains_accent(kw):
        reason.append(
            "accent"
        )

    if reason:

        suspicious.append(
            {
                "keyword": kw,
                "frequency": freq,
                "reason": "; ".join(reason)
            }
        )

# SAVE SUSPICIOUS KEYWORDS

df_suspicious = pd.DataFrame(
    suspicious
)

df_suspicious = df_suspicious.sort_values(
    by="frequency",
    ascending=False
)

df_suspicious.to_csv(
    "suspicious_keywords.csv",
    index=False,
    encoding="utf-8"
)

print(
    f"\nSuspicious keywords: "
    f"{len(df_suspicious)}"
)

print(
    "Saved: suspicious_keywords.csv"
)

# PREVIEW SUSPICIOUS TERMS

print(
    "\nTOP SUSPICIOUS KEYWORDS\n"
)

for _, row in (
    df_suspicious
    .head(100)
    .iterrows()
):

    print(
        f"{row['frequency']:5d}  "
        f"{row['keyword']} "
        f"({row['reason']})"
    )

Datasets loaded: 2255
Total keyword occurrences: 10019
Unique keywords: 2828

TOP 100 KEYWORDS

  354  biodiversity
  127  Belgium (country)
  117  Observed data
  116  Reporting INSPIRE
  108  Population
  107  Federal government
   90  Air quality
   83  Flemish Region (region)
   82  remote sensing
   78  Anvers (province)
   78  Brabant flamand (province)
   78  East Flanders (province)
   78  Flemish Brabant (province)
   78  West Flanders (province)
   78  codage, anonymisation et TTP (eHealth)
   78  codering, anonimisering en TTP (eHealth)
   78  source data at healthdata.be (Sciensano)
   76  Model data
   75  CoBRHA - Common Base Registry for HealthCare Actor (eHealth)
   72  Brabant wallon (province)
   72  Walloon Brabant (province)
   61  age
   60  Chemical composition and climate
   58  Waals-Brabant (province)
   53  atmosphere
   52  Census 2011
   52  Planetary aeronomy
   50  NIDHI number
   50  Housing
   48  infectie
   48  infection
   48  Mars
   48  National ins